# Day 11 — Column selectors

*100 Days of Polars · Phase 2: Expressions*

- **Post:** {{TODO: Medium link}}
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [1]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


`polars.selectors` (imported as `cs` by convention) are reusable column pickers you can **combine with set operations**: union, intersection, difference, complement.

In [2]:
import polars.selectors as cs

products = pl.read_csv("data/raw/olist_products_dataset.csv")
orders = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)
products.head(3)

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
str,str,i64,i64,i64,i64,i64,i64,i64
"""1e9e8ef04dbcff4541ed26657ea517…","""perfumaria""",40,287,1,225,16,10,14
"""3aa071139cb16b67ca9e5dea641aaa…","""artes""",44,276,1,1000,30,18,20
"""96bd76ec8810374ed1b65e29197571…","""esporte_lazer""",46,250,1,154,18,9,15


## 1. Selectors by dtype

In [3]:
products.select(cs.numeric()).head(3)

product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
i64,i64,i64,i64,i64,i64,i64
40,287,1,225,16,10,14
44,276,1,1000,30,18,20
46,250,1,154,18,9,15


In [4]:
orders.select(cs.string()).head(3)

order_id,customer_id,order_status
str,str,str
"""e481f51cbdc54678b7cc49136f2d6a…","""9ef432eb6251297304e76186b10a92…","""delivered"""
"""53cdb2fc8bc7dce0b6741e21502734…","""b0830fb4747a6c6d20dea0b8c802d7…","""delivered"""
"""47770eb9100c2d0c44946d9cf07ec6…","""41ce2a54c0b03bf3443c3d931a3670…","""delivered"""


In [5]:
orders.select(cs.temporal()).head(3)

order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs]
2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


## 2. Selectors by name

In [6]:
products.select(cs.ends_with("_cm")).head(3)

product_length_cm,product_height_cm,product_width_cm
i64,i64,i64
16,10,14
30,18,20
18,9,15


In [7]:
products.select(cs.starts_with("product_") & cs.contains("lenght")).head(3)

product_name_lenght,product_description_lenght
i64,i64
40,287
44,276
46,250


In [8]:
products.select(cs.first(), cs.last()).head(3)

product_id,product_width_cm
str,i64
"""1e9e8ef04dbcff4541ed26657ea517…",14
"""3aa071139cb16b67ca9e5dea641aaa…",20
"""96bd76ec8810374ed1b65e29197571…",15


## 3. Set operations

| Operator | Meaning |
|---|---|
| `a \| b` | columns in a **or** b |
| `a & b` | columns in a **and** b |
| `a - b` | columns in a but **not** b |
| `~a` | every column **not** in a |

In [9]:
# Numeric columns except the physical dimensions
products.select(cs.numeric() - cs.ends_with("_cm", "_g")).head(3)

product_name_lenght,product_description_lenght,product_photos_qty
i64,i64,i64
40,287,1
44,276,1
46,250,1


In [10]:
# Everything that is not numeric
products.select(~cs.numeric()).head(3)

product_id,product_category_name
str,str
"""1e9e8ef04dbcff4541ed26657ea517…","""perfumaria"""
"""3aa071139cb16b67ca9e5dea641aaa…","""artes"""
"""96bd76ec8810374ed1b65e29197571…","""esporte_lazer"""


In [11]:
# What does a selector resolve to on this frame?
cs.expand_selector(products, cs.numeric() - cs.ends_with("_cm", "_g"))

('product_name_lenght', 'product_description_lenght', 'product_photos_qty')

## 4. Selectors work wherever columns are expected

In [12]:
# In expressions
products.with_columns(cs.ends_with("_cm").cast(pl.Float32)).schema

Schema([('product_id', String),
        ('product_category_name', String),
        ('product_name_lenght', Int64),
        ('product_description_lenght', Int64),
        ('product_photos_qty', Int64),
        ('product_weight_g', Int64),
        ('product_length_cm', Float32),
        ('product_height_cm', Float32),
        ('product_width_cm', Float32)])

In [13]:
# In drop
products.drop(cs.contains("lenght")).columns

['product_id',
 'product_category_name',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

In [14]:
# In aggregation: null count of every numeric column
products.select(cs.numeric().null_count())

product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
u32,u32,u32,u32,u32,u32,u32
610,610,610,2,2,2,2


## 5. Gotcha: `& | - ~` are set operations only between selectors

Between two selectors, `&`, `|`, `-` and `~` combine **column sets**. As soon as one side is an ordinary expression (`pl.col(...)`, or a selector you already did maths on), they act on the **data** instead.

Polars 2.0 made this stricter: `pl.col(...)` is **no longer coerced to a selector** in `&`, `|`, `^` ([upgrade guide](https://docs.pola.rs/releases/upgrade/2/#stop-coercing-plcol-to-a-selector-in-selector)).

In [15]:
# Intended: "numeric columns that are also named product_photos_qty" -> 1 column
# Actual in 2.0: for every numeric column c, compute c & product_photos_qty row by row
trap = products.select(cs.numeric() & pl.col("product_photos_qty"))
print(trap.shape)   # 7 columns, not 1
trap.head(3)

(32951, 7)


product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
i64,i64,i64,i64,i64,i64,i64
0,1,1,1,0,0,0
0,0,1,0,0,0,0
0,0,1,0,0,1,1


In [16]:
# The real values, for comparison
products.select(cs.numeric()).head(3)

product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
i64,i64,i64,i64,i64,i64,i64
40,287,1,225,16,10,14
44,276,1,1000,30,18,20
46,250,1,154,18,9,15


All the numeric columns are `Int64`, and `&` on integers is a **bitwise AND**: it compares the binary digits.

| `product_description_lenght` | `product_photos_qty` | `&` result |
|---|---|---|
| 287 = `100011111` | 1 = `000000001` | 1 |
| 276 = `100010100` | 1 = `000000001` | 0 |

No error, no warning, and the column names still look right. The upgrade guide calls it silently wrong for compatible dtypes.

**Fix:** keep both sides selectors with `cs.by_name`.

In [17]:
products.select(cs.numeric() & cs.by_name("product_photos_qty")).head(3)  # 1 column, real values

product_photos_qty
i64
1
1
1


In [18]:
products.select(cs.numeric() - cs.by_name("product_photos_qty")).columns

['product_name_lenght',
 'product_description_lenght',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

### Arithmetic turns a selector into an expression

`cs.numeric() * 3` works as expected: it expands to every numeric column and multiplies each by 3. But what you hold afterwards is a plain expression, not a selector, so set operations no longer apply to it.

In [19]:
# Once you do arithmetic on a selector it becomes a normal expression
print(cs.is_selector(cs.numeric()))
print(cs.is_selector(cs.numeric() * 2))

True
False


In [20]:
products.select(cs.numeric() * 3).head(3)   # 7 columns, same names, values x3

product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
i64,i64,i64,i64,i64,i64,i64
120,861,3,675,48,30,42
132,828,3,3000,90,54,60
138,750,3,462,54,27,45


In [21]:
# Pick columns first, then do the maths
products.select((cs.numeric() - cs.ends_with("_cm")) * 3).columns

['product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g']

In [22]:
# The other order: maths first, then "-" on an expression -> TypeError (Expr - Selector)
try:
    out = products.select((cs.numeric() * 3) - cs.ends_with("_cm"))
    print(out.shape, out.columns)
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

TypeError -> unsupported operand type(s) for op: ('Expr' - 'Selector')


**Rule:** do all the column picking with selectors first (`cs.*`, `cs.by_name`, `& | - ~`), then chain the maths. If you are unsure what you are holding, `cs.is_selector(x)` tells you.

## Try it yourself

1. Select every `orders` column that is temporal **and** contains `delivered`.
2. Fill nulls with 0 in every numeric `products` column except `product_weight_g`, using one selector.
3. Compare `cs.matches("^product_.*_cm$")` with `pl.col("^product_.*_cm$")`. When would you prefer the selector?